# Pokémon battle prediction · 300 features

## Summary

**Result:** Soft voting reaches **67.88%** mean grouped cross-validation accuracy. The development-fit set contains **300 predictors** with maximum absolute pair correlation **0.900**.

Predict the battle winner from roster information and the first 30 observed turns.
Each model uses **300 selected predictors** with descriptive names. The notebook
loads data, removes redundant and low-utility inputs, compares four models, and
checks their validation predictions. Run all cells using the project’s Python environment.


## Context & Methods

Use only the supplied roster, opponent lead and turns 1–30. Outcome labels and
battle IDs are excluded from predictors. Five stratified folds group identical
observations; the reserved holdout and its matching duplicates are excluded.

Feature selection, imputation and scaling fit **inside each training fold**.
Accuracy ranks the models; other metrics and calibration check their behavior.
Development scores remain exploratory because model choices reuse these data.

### Key assumptions

Last-seen bench states can be stale. Some team-resource summaries presume unseen
slots are healthy; unknown opponent metadata stays missing in matchup estimates.
HP changes, hazard use and speed/damage calculations are observational proxies.
Names and definitions identify the time window, units and applicable assumption.


In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, HTML, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'src/pokemon_challenge').is_dir() and (p/'pyproject.toml').exists())
sys.path[:0] = [str(ROOT/'src'), str(ROOT/'notebooks/amirhosein')]
from pokemon_challenge.paths import data_dir
from modeling import ReducedBattleStudy, model_candidates

FEATURE_COUNT = 300
CORRELATION_LIMIT = 0.90
DATA_DIR = data_dir()
RUN = datetime.now().strftime('%Y%m%d_%H%M%S_features300')
plt.rcParams.update({'font.size': 11, 'figure.figsize': (9, 4),
                     'axes.spines.top': False, 'axes.spines.right': False})


## Data

Inputs are `train.jsonl` and the supplied `type_chart_gen2.csv` in `data/raw/`,
or the folder selected by `POKEMON_DATA_DIR`. The loader verifies numeric schemas
and constructs duplicate-aware validation partitions. Test data is read only if
the optional submission step is enabled.


In [ ]:
study = ReducedBattleStudy(data_path=DATA_DIR, owner='amirhosein', run=RUN,
                          n_features=FEATURE_COUNT, correlation_threshold=CORRELATION_LIMIT)
display(pd.DataFrame([{
    'Training battles': len(study.y), 'Development battles': len(study.development),
    'Reserved holdout': len(study._holdout),
    'Quarantined duplicates': study.manifest['quarantined_rows'],
    'Development P1 win rate': study.y.iloc[study.development].mean(),
}]).round(3))


## Select 300 predictors

Remove all-missing and constant columns, then keep one representative of each
duplicate. Rank the remaining measurements using a blend of regularized logistic
coefficients, shallow-tree importance and univariate association. Greedily retain
the most useful columns whose absolute Pearson correlation stays below the limit.
Stop at exactly **300**; do not relax the limit or restore redundant columns.

The selection below fits development data to provide the final readable feature
list. Validation refits this procedure on each training fold, so individual fold
selections can differ. Utility is a ranking heuristic, not causal importance.


In [ ]:
selected_features = study.select_features()
selector = study.feature_selector
selection_audit = selector.audit_
selection_summary = (selection_audit.reason.value_counts().rename_axis('Decision')
                     .rename('Features').to_frame())
display(selection_summary)
print(f'Model predictors: {selected_features.shape[1]}; '
      f'maximum retained |correlation|: {selector.maximum_selected_correlation_:.3f}.')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
removed = selection_summary.drop(index='selected', errors='ignore').sort_values('Features')
bars = axes[0].barh(removed.index.str.replace('_', ' '), removed.Features, color='#8B99A5')
axes[0].bar_label(bars, padding=3)
axes[0].set(xlabel='Removed predictors', title='Redundancy and utility filtering')
axes[0].set_xlim(0, removed.Features.max()*1.18)
category_counts = study.feature_definitions.category.value_counts()
counts = category_counts.head(8).copy()
if len(category_counts) > 8:
    counts.loc['Other measurements'] = category_counts.iloc[8:].sum()
counts = counts.sort_values()
axes[1].barh(counts.index, counts, color='#356A95')
axes[1].set(xlabel='Selected predictors', title='The 300-feature set')
fig.tight_layout()
fig.savefig(study.output_path/'feature_selection.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
definitions = study.feature_definitions
display(HTML('<details><summary>Definitions of all 300 predictors</summary>'
             + definitions.to_html(index=False, escape=True) + '</details>'))


## Compare models

Compare logistic regression, histogram boosting, XGBoost and an equal-weight
soft-voting ensemble on the same folds. Each training pipeline selects exactly
300 inputs. Fold standard deviation describes variation across folds, not a
confidence interval. Detailed fold predictions and settings are saved with the run.


In [ ]:
results = study.compare(model_candidates())
MODEL_LABELS = {'logistic': 'Logistic regression', 'histogram_boosting': 'Histogram boosting',
                'xgboost': 'XGBoost', 'soft_voting': 'Soft voting'}
comparison = results[['candidate', 'input_features', 'cv_accuracy', 'fold_std',
                      'roc_auc', 'log_loss', 'train_cv_gap']].copy()
comparison['candidate'] = comparison.candidate.map(MODEL_LABELS)
display(comparison.rename(columns={'candidate': 'Model', 'input_features': 'Features',
    'cv_accuracy': 'CV accuracy', 'fold_std': 'Fold SD', 'roc_auc': 'ROC AUC',
    'log_loss': 'Log loss', 'train_cv_gap': 'Train–CV gap'}).round(4))

ranked = results.sort_values('cv_accuracy')
fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.barh(ranked.candidate.map(MODEL_LABELS), ranked.cv_accuracy, color='#356A95')
ax.errorbar(ranked.cv_accuracy, np.arange(len(ranked)), xerr=ranked.fold_std,
            fmt='none', ecolor='#263746', capsize=3)
ax.bar_label(bars, labels=[f'{value:.2%}' for value in ranked.cv_accuracy], padding=10)
ax.axvline(results.dummy_accuracy.iloc[0], color='#666666', linestyle='--', label='Majority reference')
ax.set(xlim=(0, 1), xlabel='Development accuracy', title='300-feature models · five grouped folds')
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc='lower right', frameon=False)
fig.tight_layout()
fig.savefig(study.output_path/'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## Check predictions

Out-of-fold predictions come from models that did not train on the corresponding
observation group. The confusion matrix shows mistakes at each model’s decision
rule; the calibration curve compares predicted and observed win probabilities.
Selecting the best model on these results remains adaptive.


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

best_name = study.best_candidate
truth = study.y.iloc[study.development].to_numpy()
probability = study.oof_probabilities[best_name]
predictions = study.oof_predictions[best_name]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
matrix = confusion_matrix(truth, predictions, labels=[0, 1])
ConfusionMatrixDisplay(matrix, display_labels=['P1 loss', 'P1 win']).plot(
    ax=axes[0], colorbar=False, cmap='Blues', values_format='d')
axes[0].set_title('Out-of-fold decisions')
observed, predicted = calibration_curve(truth, probability, n_bins=8, strategy='quantile')
axes[1].plot([0, 1], [0, 1], '--', color='#666666', label='Perfect calibration')
axes[1].plot(predicted, observed, 'o-', color='#356A95', label=MODEL_LABELS[best_name])
axes[1].set(xlabel='Mean predicted P1 win probability', ylabel='Observed P1 win fraction',
            title='Out-of-fold calibration', xlim=(0, 1), ylim=(0, 1))
axes[1].legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(study.output_path/'prediction_checks.png', dpi=150, bbox_inches='tight')
plt.show()


## Takeaways

The summary uses this run’s grouped validation results. The selected feature list
and the reason for every removal are saved as readable tables. A high utility score
or validation gain does not establish a causal effect.


In [ ]:
best = results.iloc[0]
summary = {'selected_features': FEATURE_COUNT, 'best_model': best_name,
    'cv_accuracy': float(best.cv_accuracy), 'fold_std': float(best.fold_std),
    'maximum_selected_correlation': selector.maximum_selected_correlation_,
    'correlation_limit': CORRELATION_LIMIT, 'development_rows': len(study.development),
    'removal_counts': selection_audit.reason.value_counts().to_dict(),
    'holdout_evaluated': False, 'test_read': False}
display(Markdown(f"**{MODEL_LABELS[best_name]}** reaches **{best.cv_accuracy:.2%}** mean "
    f"grouped CV accuracy with **300 predictors**. The development-fit feature set’s "
    f"largest absolute pair correlation is **{selector.maximum_selected_correlation_:.3f}**. "
    'No holdout evaluation or test prediction is performed by default.'))


## Optional final evaluation

Enable holdout evaluation only after choosing the model. This locks further
comparisons. Submission generation then refits the selected pipeline on all labels
and saves predictions in sample-submission order. Feature selection is refitted too;
its final 300 names are saved separately. Nothing is uploaded automatically.


In [ ]:
EVALUATE_HOLDOUT = False
MAKE_SUBMISSION = False

if EVALUATE_HOLDOUT:
    holdout = study.evaluate_holdout(best_name)
    print(f'Holdout accuracy: {holdout["accuracy"]:.2%}')
if MAKE_SUBMISSION:
    study.make_submission()

summary.update(holdout_evaluated=study.manifest['holdout_evaluated'],
               test_read=study.manifest['test_read'])
(study.output_path/'run_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
study.save_run_record(ROOT)
print('Saved feature definitions, selection audit, model settings and predictions:', study.output_path)
